In [1]:
import pandas as pd
import numpy as np

In [2]:
df_full = pd.read_csv('data_si.csv')
df_full.drop(['age', 'gender', 'insulin_milliUnit', 'insulin_milliUnit - last 2 hours'], axis=1, inplace=True)
len(df_full)

159566

In [3]:
length_threshold = 48

long_uids = []
for uid in df_full['idx'].unique():
    if len(df_full[df_full['idx'] == uid]) >= length_threshold:
        long_uids.append(int(uid))

df_long_uids = df_full.copy()
df_long_uids = df_long_uids[df_full['idx'].isin(long_uids)]

df_long_uids.head()

,idx,star_id,bg,insulin_milliUnit - last 3 hours,carbs_milliGramm,SI
0,1.0,00035d46-c466-4303-8e49-f63e85395d00,12.501408,5000.0,8884.00,0.000068
1,1.0,00035d46-c466-4303-8e49-f63e85395d00,12.157971,11000.0,7690.36,0.000065
2,1.0,00035d46-c466-4303-8e49-f63e85395d00,10.186957,17000.0,6448.00,0.000141
3,1.0,00035d46-c466-4303-8e49-f63e85395d00,9.466116,13500.0,6715.96,0.000161
4,1.0,00035d46-c466-4303-8e49-f63e85395d00,10.755372,9000.0,7422.40,0.000069


In [4]:
_, counts = np.unique(df_long_uids['idx'], return_counts=True)
print(min(counts))

48


In [4]:
for col_name in ['ds', 'unique_id']:
    if col_name in df_long_uids.columns:
        df_long_uids.drop([col_name], axis=1, inplace=True)

unique_ids = []
ds_list = []
starting_ds = pd.Timestamp(year=2020, month=1, day=1, hour=0)
j = 0
prev_uid = 0.0
for i in range(len(df_long_uids)):
    current_uid = df_long_uids.iloc[i]['idx']
    if current_uid != prev_uid:
        ds = starting_ds
    else:
        ds += pd.Timedelta(hours=1)
    ds_list.append(ds)
    unique_ids.append(f"{df_long_uids.iloc[i]['star_id'][-4:]}-{ds.month:02d}_{ds.day:02d}")
    prev_uid = current_uid

df_long_uids.loc[:, 'ds'] = ds_list
df_long_uids.loc[:, 'unique_id'] = unique_ids
df_long_uids.drop(['star_id'], axis=1, inplace=True)

In [7]:
df_long_uids.reset_index(drop=True, inplace=True)
complete_days_unique_ids = []
uids_and_counts = np.unique(df_long_uids['unique_id'].to_numpy(), return_counts=True)
for i, unique_id in enumerate(uids_and_counts[0]):
    if uids_and_counts[1][i] == 24:
        complete_days_unique_ids.append(unique_id)

df_towrite = df_long_uids.copy()
df_towrite = df_towrite[df_long_uids['unique_id'].isin(complete_days_unique_ids)]
df_towrite.reset_index(drop=True, inplace=True)
df_towrite.head()

,idx,bg,insulin_milliUnit - last 3 hours,carbs_milliGramm,SI,ds,unique_id
0,1.0,12.501408,5000.0,8884.00,0.000068,2020-01-01 00:00:00,5d00-01_01
1,1.0,12.157971,11000.0,7690.36,0.000065,2020-01-01 01:00:00,5d00-01_01
2,1.0,10.186957,17000.0,6448.00,0.000141,2020-01-01 02:00:00,5d00-01_01
3,1.0,9.466116,13500.0,6715.96,0.000161,2020-01-01 03:00:00,5d00-01_01
4,1.0,10.755372,9000.0,7422.40,0.000069,2020-01-01 04:00:00,5d00-01_01


In [9]:
_, counts = np.unique(df_long_uids['idx'], return_counts=True)
print(min(counts))

48


In [10]:
df_towrite.to_csv('hourly_ICU_for_nf.csv', sep=',')
df_long_uids.to_csv('hourly_ICU_for_nf_notfulldays.csv', sep=',')